In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [ ]:
load_dotenv(override=True)

In [ ]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

In [ ]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [ ]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [ ]:
search_description = """Use this tool to perform web search for a given query"""
planner_description = """Use this tool to plan a set of web search queries for a given topic"""
writer_description = """Use this tool to write Research Reports"""

In [ ]:
search_tool = search_agent.as_tool(tool_name="search_tool", tool_description=search_description)
planner_tool = planner_agent.as_tool(tool_name="planner_tool", tool_description=planner_description)
writer_tool = writer_agent.as_tool(tool_name="writer_tool", tool_description=writer_description)

tools = [search_tool, planner_tool, writer_tool, send_email_tool]

tools

In [ ]:
planner_tool.params_json_schema

In [ ]:
Manager_Instructions = """You are a ResearchReport Generation Manager who sends emails. You would be given a Research Topic, you will have to Plan, Search, Write and send Emails.

You have access to 4 tools:
1. planner_tool: Use this tool to plan web search queries for a given topic. Once called It will return 5 search queries to perform for the given topic.
2. search_tool: Use this tool to perform web search for a given query
3. writer_tool: Use this tool to write Research Reports
4. send_email_tool: Use this tool to send emails

You should follow this workflow:
1) Use the planner_tool which plans 5 web search queries for the given research topic. Call this tool only once at the starting of the workflow. It will give you 5 queries.
2) Use the search_tool to perform web searches for each of the planned queries. Once you get the 5 queries, call the search_tool for each of the 5 queries independently. You will get 5 search results for the 5 queries.
3) Base on the searches of all the 5 queries, send the entire context of all the 5 searches to writer_tool. This writer_tool will write a research report.
4) Once the Research report is generated, Use the send_email_tool to send the research report via email.
5) Use the send_email_tool only once, after the research report is generated. Do not send multiple emails.

Remeber the workflow:
1) Plan Once -> 2) Search 5 times -> 3) Write once -> 4) Send Email Once
"""

In [ ]:
manager = Agent(name="ResearchReportManager", instructions=Manager_Instructions, tools=tools, model=MODEL_NAME)

In [ ]:
import os
import shutil

GRAPHVIZ_BIN = r"C:\Program Files\Graphviz\bin"
if os.path.exists(GRAPHVIZ_BIN) and GRAPHVIZ_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] = GRAPHVIZ_BIN + os.pathsep + os.environ.get("PATH", "")

if shutil.which("dot"):
    from agents.extensions.visualization import draw_graph
    draw_graph(manager)
else:
    print("Graphviz 'dot' executable is still not on PATH. Restart the notebook kernel after installing Graphviz.")

In [ ]:
from agents.extensions.visualization import draw_graph
draw_graph(manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(manager, "Which Agentic Frameworks are most popular in 2026?")

In [ ]:
Markdown(result.final_output)